# docagent on Google Colab (vLLM), from GitHub

Run top to bottom on a GPU runtime (Runtime → Change runtime type → GPU). The code is cloned from
[github.com/YashThirani21/docagent](https://github.com/YashThirani21/docagent). **No setup or tokens needed.**

Where state that outlives the runtime (`store/` and the run outputs) goes is picked in cell 1 by `PERSIST`:
- `"auto"` (default): `"hf"` if this Colab account has an `HF_TOKEN` secret, otherwise `"none"`.
- `"none"`: nothing is saved between sessions. The index is built fresh, and the run outputs are downloaded to your
  computer as a zip at the end.
- `"drive"`: your own `MyDrive/ColabProjects/` (asks for Drive access once).
- `"hf"`: a private Hugging Face dataset repo, for the maintainer. Every save is a commit, so earlier stores and runs
  stay available. Needs a Colab secret `HF_TOKEN` (🔑 icon in the left bar) holding a token with write access.

## 1. Settings

In [1]:
import os, time
from pathlib import Path

REPO_URL = "https://github.com/YashThirani21/docagent.git"
REF = "main"
PROJECT = Path("/content/senior-data-scientist")

PERSIST = "auto"                # "auto", "none" (session only), "drive" (your MyDrive/ColabProjects) or "hf"
HF_DATASET = "YashThirani21/docagent-store"         # for "hf" only: created private on the first save
DRIVE_DIR = Path("/content/drive/MyDrive/ColabProjects")

RUN_ID = time.strftime("%Y%m%d-%H%M%S")             # run outputs are saved under runs/<RUN_ID>/

if PERSIST in ("auto", "hf"):
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        PERSIST = "hf"
    except Exception as e:      # no such secret, or notebook access to it not granted
        if PERSIST == "hf":
            raise RuntimeError(
                f"No usable Colab secret HF_TOKEN ({type(e).__name__}). Add a token with Write access in "
                "Colab's 🔑 Secrets panel as HF_TOKEN with 'Notebook access' on, or set PERSIST = \"none\"."
            ) from None
        PERSIST = "none"
if PERSIST == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
elif PERSIST not in ("hf", "none"):
    raise ValueError(f"PERSIST must be 'auto', 'none', 'drive' or 'hf', not {PERSIST!r}")
print(f"persist: {PERSIST}, run: {RUN_ID}")

persist: none, run: 20261020-030032


## 2. Clone the project

In [2]:
!rm -rf {PROJECT}
!git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}
!git checkout -q {REF} && git log -1 --format="running %h  %s  (%cd)" --date=short

/content/senior-data-scientist
running 1ff0923  Created using Colab  (2026-10-20)


## 3. Save / restore helpers

Every read and write of saved state goes through these three functions (no-ops or local downloads when
`PERSIST` is `"none"`), so switching `PERSIST` changes nothing
else in the notebook. Hugging Face calls run through the `hf` CLI with `HF_HUB_OFFLINE=0`, because the docagent
config puts the kernel in offline mode once it is loaded (`privacy.offline`).

In [3]:
import shutil, subprocess

STORE_ARCHIVE = Path("/content/store-backup.tar.gz")
RUN_ITEMS = ["outputs", "eval/results", "store/traces.db", "store/logs", "vllm.log"]

def _hf(*args):
    env = {**os.environ, "HF_HUB_OFFLINE": "0"}
    return subprocess.run(["hf", *args, "--repo-type", "dataset"], env=env, capture_output=True, text=True)

def pull_store():
    """Restore store/ from the last saved backup; on the first run there is none."""
    STORE_ARCHIVE.unlink(missing_ok=True)
    if PERSIST == "none":
        print("Session-only run: starting with an empty store; cell 10 builds the index.")
        return
    if PERSIST == "hf":
        # Fresh folder: if the Hub can't be reached, `hf download` silently returns a file already in --local-dir
        pull_dir = Path("/content/hf_pull")
        shutil.rmtree(pull_dir, ignore_errors=True)
        r = _hf("download", HF_DATASET, STORE_ARCHIVE.name, "--local-dir", str(pull_dir))
        found = r.returncode == 0 and (pull_dir / STORE_ARCHIVE.name).is_file()
        if found:
            shutil.move(pull_dir / STORE_ARCHIVE.name, STORE_ARCHIVE)
        error = next((l for l in r.stderr.splitlines() if l.startswith("Error")), r.stderr.strip()[:200])
        missing = f"{HF_DATASET}/{STORE_ARCHIVE.name} ({error})"
    else:
        found = (DRIVE_DIR / STORE_ARCHIVE.name).is_file()
        if found:
            shutil.copy2(DRIVE_DIR / STORE_ARCHIVE.name, STORE_ARCHIVE)
        missing = DRIVE_DIR / STORE_ARCHIVE.name
    if not found:
        # First run: cell 10 builds the index from scratch, cell 12 saves the backup.
        print(f"No backup at {missing} - starting with an empty store (first run).")
        return
    subprocess.run(["tar", "-xzf", str(STORE_ARCHIVE), "-C", str(PROJECT)], check=True)
    print("Store restored successfully.")

def push_store():
    """Save store/ as one archive; on HF each save is a new commit of the same file."""
    if PERSIST == "none":
        print("Session-only run: store/ is not saved (it is rebuilt by cell 10 next time).")
        return
    subprocess.run(["tar", "-czf", str(STORE_ARCHIVE), "-C", str(PROJECT), "store"], check=True)
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(STORE_ARCHIVE), STORE_ARCHIVE.name, "--private",
                "--commit-message", f"store backup {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        where = f"hf.co/datasets/{HF_DATASET}"
    else:
        shutil.copy2(STORE_ARCHIVE, DRIVE_DIR / STORE_ARCHIVE.name)
        where = DRIVE_DIR / STORE_ARCHIVE.name
    print(f"Backup saved: {where}")
    print(f"Archive size: {STORE_ARCHIVE.stat().st_size / (1024**3):.2f} GB")

def push_run():
    """Save outputs, eval results, traces and logs under runs/<RUN_ID>/."""
    staging = Path("/content/run_out") / RUN_ID
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    for item in RUN_ITEMS:
        p = PROJECT / item
        if p.is_dir():
            shutil.copytree(p, staging / p.name)
        elif p.is_file():
            shutil.copy2(p, staging / p.name)
        else:
            print(f"skipped (not found): {item}")
    if PERSIST == "hf":
        r = _hf("upload", HF_DATASET, str(staging), f"runs/{RUN_ID}", "--private",
                "--commit-message", f"run {RUN_ID}")
        if r.returncode:
            raise RuntimeError(r.stderr)
        print(f"Run saved: hf.co/datasets/{HF_DATASET}/tree/main/runs/{RUN_ID}")
    elif PERSIST == "none":
        from google.colab import files
        zip_path = shutil.make_archive(f"/content/docagent-run-{RUN_ID}", "zip", staging)
        files.download(zip_path)
        print(f"Run downloaded to your computer: {Path(zip_path).name}")
    else:
        shutil.copytree(staging, DRIVE_DIR / "run_out" / RUN_ID, dirs_exist_ok=True)
        print(f"Run saved: {DRIVE_DIR / 'run_out' / RUN_ID}")

## 4. Restore `store/` from the last backup (skipped on the first run)

In [4]:
pull_store()

Session-only run: starting with an empty store; cell 10 builds the index.


## 5. Install vLLM and the docagent package

In [ ]:
%cd /content/senior-data-scientist

!pip install -q vllm==0.31.0 -e . -c requirements-gpu.txt

import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import docagent, vllm; print('docagent and vllm', vllm.__version__, 'installed')"],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Install failed; rerun this cell without -q to see pip's error.\n" + r.stderr[-1500:])
print(r.stdout.strip())

/content/senior-data-scientist
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

## 6. Check the GPU

In [ ]:
import shutil, subprocess
# vLLM and the bge-m3 embedder (device: cuda in the colab configs) both need a GPU; stop before cells 7-10 fail
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi"], capture_output=True).returncode:
    raise RuntimeError("No GPU in this session. Runtime → Change runtime type → T4 GPU (or L4/A100), "
                       "then Runtime → Run all. If no GPU is offered, Colab's free GPU quota is used up for now.")
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 7. Pick the config and start the matching vLLM server

In [ ]:
import os, yaml
%cd /content/senior-data-scientist

# Pick the agent version here; the vLLM server is started to match it.
CONFIG = "configs/colab_vllm_awq.yaml"   # or "configs/colab_vllm.yaml"
# CONFIG = "configs/colab_vllm.yaml"
MAX_LEN = 8192                           # vLLM --max-model-len (16384 if prompts get rejected for length)
GPU_UTIL = 0.90                          # share of GPU memory for vLLM; the rest is for bge-m3 on cuda

os.environ["DOCAGENT_CONFIG"] = CONFIG   # CLI, chat app and Agent() all read this
SERVED = yaml.safe_load(open(CONFIG))["llm"]["model"]   # vLLM must serve exactly the name the config asks for

print(SERVED)
if SERVED.endswith("-awq"):
    # AWQ 4-bit weights in fp16: fits a T4
    print(f"{CONFIG}: serving AWQ as {SERVED}")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} MODEL=Qwen/Qwen2.5-VL-7B-Instruct-AWQ QUANT=awq DTYPE=half SERVED={SERVED} bash scripts/start_vllm.sh
else:
    # start_vllm.sh picks the weights for the GPU: bf16 on L4/A100, AWQ on a T4
    print(f"{CONFIG}: serving as {SERVED}, weights chosen by GPU")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} SERVED={SERVED} bash scripts/start_vllm.sh

## 8. Check the server responds

In [ ]:
!curl -s http://127.0.0.1:8000/v1/models | head -c 200

## 9. Download the embedding model (bge-m3)

In [ ]:
!python scripts/download_models.py

## 10. Ingest the reports

In [ ]:
!python -m docagent.cli ingest data/reports
!python -m docagent.cli status

## 11. Launch the chat app (public Gradio share link)

In [ ]:
import sys
sys.path.insert(0, "/content/senior-data-scientist/src")
sys.path.insert(0, "/content/senior-data-scientist/app")

import chat
from IPython.display import HTML, display

demo = chat.demo

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=True,
    prevent_thread_lock=True,
    inline=False,
)

url = demo.share_url
display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))

## 12. Save `store/` (Hugging Face or Drive, per `PERSIST`; skipped when `"none"`)

In [ ]:
push_store()

## 13. Save run outputs, eval results, traces and logs

In [ ]:
push_run()

## 14. Trace of the latest request

In [ ]:
!python -m docagent.cli trace last

## 15. Spans of one request (replace the request id)

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("store/traces.db")
pd.read_sql("select * from spans where request_id = '53b8c2c82def' order by started_at", con)

## 16. Ask a few questions in the notebook

In [ ]:
import sys, time
sys.path.insert(0, "/content/senior-data-scientist/src")
from IPython.display import Markdown, display
from docagent.agent import Agent
from docagent.render import render_markdown

questions = [
    "What is the fatigue life of a 2024-T3 spar in cycles?",
    "Why did the left wing main spar fail?",
    "What caused the throttle arm to fracture?",
]
agent = chat.agent if "chat" in globals() else Agent()
batch_id = "questions-" + time.strftime("%Y%m%d-%H%M%S")
for q in questions:
    resp = agent.ask(q, batch_id=batch_id)
    display(Markdown(f"### {q}\n\n" + render_markdown(resp)))
print("traces:", batch_id)

## Reading saved runs back

On Hugging Face, fetch any earlier run, or the store as it was at any commit (see the repo's *Files → History*):
```python
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} --repo-type dataset --include "runs/<RUN_ID>/*" --local-dir /content/saved
!HF_HUB_OFFLINE=0 hf download {HF_DATASET} store-backup.tar.gz --repo-type dataset --revision <commit> --local-dir /content
```
With `"none"` each run is the downloaded `docagent-run-<RUN_ID>.zip`. On Drive they are in `MyDrive/ColabProjects/run_out/<RUN_ID>/`.